In [ ]:
import re
import torch


def time_to_seconds(x):
    d, hms = x.split("_")
    h, m, s = map(int, hms.split(":"))
    return h * 3600 + m * 60 + s


def parse_intervals(line):
    matches = re.findall(r"\((.*?) (.*?)\)", line)
    intervals = []
    for s, e in matches:
        intervals.append([time_to_seconds(s), time_to_seconds(e)])

    return intervals


def parse_problem(text):
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    idx = 0
    assert lines[idx] == "TOOLS"
    idx += 1

    n_tools = int(lines[idx])
    idx += 1
    tools = []

    for _ in range(n_tools):
        tools.append(parse_intervals(lines[idx]))
        idx += 1

    assert lines[idx] == "OPERATIONS"
    idx += 1

    n_ops = int(lines[idx])
    idx += 1

    interruptible = list(map(int, lines[idx].split()))
    idx += 1

    assert lines[idx] == "TIMES"
    idx += 1

    times = []

    for _ in range(n_ops):

        row = [time_to_seconds(x) for x in lines[idx].split()]
        times.append(row)

        idx += 1

    times = torch.tensor(times).float()

    jobs = []

    while idx < len(lines):

        if lines[idx] != "work":
            idx += 1
            continue

        idx += 1

        release, deadline, penalty, job_id, n_edges, n_nodes = lines[idx].split()

        idx += 1

        nodes = list(map(int, lines[idx].split()))
        idx += 1

        edges = []

        for _ in range(int(n_edges)):
            u, v = map(int, lines[idx].split())
            edges.append((u, v))
            idx += 1

        jobs.append({"nodes": nodes, "edges": edges, "penalty": float(penalty)})

    return {
        "tools": tools,
        "times": times,
        "interruptible": torch.tensor(interruptible),
        "jobs": jobs,
    }


import torch


def build_operation_features(times, interruptible):
    feats = []
    for i in range(times.shape[0]):
        row = times[i]
        valid = row[row > 0]
        if len(valid) == 0:
            feats.append(torch.zeros(6))
            continue

        feats.append(
            torch.tensor(
                [
                    valid.min(),
                    valid.mean(),
                    valid.max(),
                    valid.std(),
                    len(valid),
                    interruptible[i],
                ]
            )
        )

    return torch.stack(feats)


def build_edge_index(jobs):
    edges = []
    for job in jobs:
        edges += job["edges"]

    if len(edges) == 0:
        return torch.zeros((2, 0)).long()

    return torch.tensor(edges).t().long()


import os
import pandas as pd
import torch
from torch.utils.data import Dataset


class SchedulingDataset(Dataset):
    def __init__(self, csv_file, problems_dir):
        self.df = pd.read_csv(csv_file)
        self.dir = problems_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        fname = row["file_path"]
        path = os.path.join(self.dir, fname)

        with open(path) as f:
            text = f.read()

        problem = parse_problem(text)
        times = problem["times"]
        interruptible = problem["interruptible"]
        op_features = build_operation_features(times, interruptible)
        edge_index = build_edge_index(problem["jobs"])
        label = torch.tensor(row[1:].to_list()).float()

        return {
            "tools": problem["tools"],
            "op_features": op_features,
            "edge_index": edge_index,
            "label": label,
        }


def collate_fn(batch):
    return batch


import torch
import torch.nn as nn
from torch_geometric.nn import TransformerConv


class ScheduleEncoder(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.input = nn.Linear(2, d)
        layer = nn.TransformerEncoderLayer(d_model=d, nhead=4, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, 2)
        self.pool = nn.Linear(d, 1)

    def forward(self, x):
        x = self.input(x)
        x = self.encoder(x)
        w = torch.softmax(self.pool(x), dim=1)

        return (x * w).sum(1)


class SchedulingModel(nn.Module):
    def __init__(self, n_algorithms, d=128):
        super().__init__()
        self.schedule = ScheduleEncoder(d)

        self.op_encoder = nn.Sequential(nn.Linear(6, d), nn.ReLU(), nn.Linear(d, d))

        self.graph = TransformerConv(d, d)
        self.cross = nn.MultiheadAttention(d, 4, batch_first=True)

        self.pool = nn.Linear(d, 1)

        self.head = nn.Linear(d, n_algorithms)

    def encode_tools(self, tools):

        emb = []

        for tool in tools:

            x = torch.tensor(tool).float()

            durations = x[:, 1] - x[:, 0]

            x = torch.stack([x[:, 0], durations], dim=1)

            x = x.unsqueeze(0)

            emb.append(self.schedule(x))

        return torch.cat(emb)

    def forward(self, problem):

        tool_emb = self.encode_tools(problem["tools"])

        op_emb = self.op_encoder(problem["op_features"])

        if problem["edge_index"].numel() > 0:

            op_emb = self.graph(op_emb, problem["edge_index"])

        op_emb = op_emb.unsqueeze(0)

        tool_emb = tool_emb.unsqueeze(0)

        op_ctx, _ = self.cross(op_emb, tool_emb, tool_emb)

        nodes = torch.cat([op_ctx.squeeze(0), tool_emb.squeeze(0)])

        w = torch.softmax(self.pool(nodes), dim=0)

        problem_emb = (nodes * w).sum(0)

        return self.head(problem_emb)

In [ ]:
import torch
from torch.utils.data import DataLoader


dataset = SchedulingDataset(
    "labels.csv",
    ""
)

loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=lambda x: x
)

sample = dataset[0]
n_algorithms = sample["label"].shape[0]

model = SchedulingModel(n_algorithms)

opt = torch.optim.AdamW(
    model.parameters(),
   lr=3e-4
)

loss_fn = torch.nn.BCEWithLogitsLoss()

from tqdm import tqdm

for epoch in range(20):
    pbar = tqdm(loader)
    
    for batch in pbar:
        problem = batch[0]
        pred = model(problem)
        target = problem["label"]
        loss = loss_fn(pred, target)
        opt.zero_grad()
        loss.backward()
        opt.step()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    print("epoch", epoch, "loss", loss.item())

In [ ]:
with open('build/samples/micro_sample.txt') as f: text = f.read()
print(parse_problem(text))